<a href="https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/%E6%B1%BA%E7%AD%96%E6%A8%B9%E9%9B%86%E6%88%90%E6%A8%A1%E5%9E%8B/randomForestEnsemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
# 💡 環境檢查與 Colab 自動化設定 (Colab Setup)
import os
import urllib.request

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # 自動安裝 Colab 缺少之套件
    !pip install -q mglearn graphviz
    
    # 自動下載中文字型檔案 (ChineseFont.ttf)
    font_url = "https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/source_data/ChineseFont.ttf"
    if not os.path.exists("ChineseFont.ttf"):
        urllib.request.urlretrieve(font_url, "ChineseFont.ttf")


## 🌲 Decision Tree Ensembles（決策樹集成模型）

單一決策樹的主要缺點是**極易對訓練資料產生過擬合（Overfitting）**。為了克服這個問題，機器學習引進了**集成學習（Ensemble Learning）**的概念。

集成模型透過組合**多個機器學習模型（通常是幾十到上百棵決策樹）**共同做出預測，目標是構建一個比單一模型更準確、更穩定的強模型。

### 2.3.1 集成模型的三大主要類型（Bagging, Boosting, Stacking）

| 集成類型 | 代表模型 | 運作原理 | 主要優點 |
| :--- | :--- | :--- | :--- |
| **Bagging (裝袋法)** | **隨機森林 (Random Forest)** | 同時建立多棵**獨立**決策樹，各自抽樣數據與特徵，最後**投票或平均**。 | 大幅降低過擬合，抗雜訊能力強 |
| **Boosting (提升法)** | **GBDT, XGBoost, LightGBM** | **循序 (Sequential)** 建立決策樹，每一棵樹專注於**修正上一棵樹的錯誤**。 | 預測準確率極高，適合複雜數據 |
| **Stacking (堆疊法)** | 自訂混合模型 | 結合不同演算法（如 RF + GBDT + SVM），將第一層預測做為新特徵訓練第二層模型。 | 融合多種模型優點 |

### 2.3.2 隨機森林（Random Forests）原理

隨機森林背後的核心思想是：每棵樹可能在部分資料上記住雜訊並產生過擬合，但如果我們**建立許多棵隨機化且互不相同的決策樹**，透過平均它們的預測結果，過擬合的個案會互相抵銷，從而大幅減少整體模型的過擬合量。

#### 隨機森林的兩種「隨機化」機制：
1. **Bootstrap 抽樣（資料點隨機化）**：
   從 $N$ 個訓練樣本中，**有放回地重複隨機抽樣 $N$ 次**。這會產生與原資料集大小相同的新資料集，但大約有 1/3 的樣本未被抽中（稱為 Out-of-Bag 樣本），部分樣本則重複出現。
2. **特徵子集隨機化 (`max_features`)**：
   在每個節點尋找最佳切割時，演算法不會檢查所有特徵，而是**隨機挑選一部分特徵（控制於 `max_features` 參數）**，從中尋找最佳切分點。這確保了每棵樹使用的特徵組合都不盡相同。

### 2.3.3 分析隨機森林：5 棵樹的決策邊界（📌 對應圖 2-33）

我們在 `two_moons` 雙月資料集中建立一個由 **5 棵決策樹** 組成的隨機森林，並將每一棵獨立樹學到的決策邊界，與整個隨機森林最終合成的決策邊界進行視覺化比較（📌 **對應圖 2-33**）：

In [ ]:
# 📌 標註：圖 2-33 —— 5 棵樹隨機森林獨立決策邊界與整體預測邊界
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import mglearn
import matplotlib.pyplot as plt

X, y = make_moons(n_samples=100, noise=0.25, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# 建立由 5 棵樹組成的隨機森林
forest = RandomForestClassifier(n_estimators=5, random_state=2)
forest.fit(X_train, y_train)

# 繪製 5 棵獨立樹的邊界與最終組合邊界
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for i, (ax, tree) in enumerate(zip(axes.ravel(), forest.estimators_)):
    ax.set_title(f"樹 {i+1}")
    mglearn.plots.plot_tree_partition(X_train, y_train, tree, ax=ax)

mglearn.plots.plot_2d_separator(forest, X_train, fill=True, ax=axes[-1, -1], alpha=0.4)
axes[-1, -1].set_title("隨機森林整體預測邊界")
mglearn.discrete_scatter(X_train[:, 0], X_train[:, 1], y_train, ax=axes[-1, -1])
plt.tight_layout()
plt.show()

#### 📌 圖 2-33 結果觀察：
* 5 棵單一決策樹的邊界各自不同（因為使用了不同的 Bootstrap 樣本與隨機特徵），且個別樹的邊界依然較為生硬曲折。
* 當將 5 棵樹的預測以軟投票（Soft Voting，平均機率）合體後（右下圖），**隨機森林的整體決策邊界變得平滑且自然得多**，能很好地泛化並區分兩個類別！

### 2.3.4 乳癌資料集實作：100 棵樹組成的隨機森林

現在我們在更複雜的乳癌資料集（Breast Cancer）上套用由 **100 棵決策樹（`n_estimators=100`）** 組成的隨機森林模型，並比較其與單一決策樹的表現差異：

In [ ]:
# 100 棵樹組成的隨機森林在乳癌資料集上的表現
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=0
)

forest = RandomForestClassifier(n_estimators=100, random_state=0)
forest.fit(X_train, y_train)

print(f"隨機森林 訓練集準確率：{forest.score(X_train, y_train):.3f}")
print(f"隨機森林 測試集準確率：{forest.score(X_test, y_test):.3f}")

在未經任何複雜微調的情況下，隨機森林在測試集上達到了 **97.2%** 的高準確率（單一未剪枝決策樹為 93.7%，單一預剪枝決策樹為 95.1%）。這證實了隨機森林在預設參數下通常就能運作得相當優秀。

### 2.3.5 隨機森林的特徵重要性（Feature Importances，📌 對應圖 2-34）

與單一決策樹類似，隨機森林同樣提供特徵重要性，它是透過**聚合森林中所有樹的特徵重要性取平均**計算得出。

由於結合了許多樹的視角，隨機森林計算出的特徵重要性比單一樹更加平滑、穩定且可靠（📌 **對應圖 2-34**）：

In [ ]:
# 📌 標註：圖 2-34 —— 乳癌資料集隨機森林特徵重要性直方圖 (繁體中文標籤)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

my_font = fm.FontProperties(fname="ChineseFont.ttf")

zh_feature_names = [
    "平均半徑", "平均紋理", "平均周長", "平均面積", "平均平滑度",
    "平均緊密度", "平均凹陷度", "平均凹陷點數", "平均對稱性", "平均分形維度",
    "半徑標準誤", "紋理標準誤", "周長標準誤", "面積標準誤", "平滑度標準誤",
    "緊密度標準誤", "凹陷度標準誤", "凹陷點數標準誤", "對稱性標準誤", "分形維度標準誤",
    "最大半徑", "最大紋理", "最大周長", "最大面積", "最大平滑度",
    "最大緊密度", "最大凹陷度", "最大凹陷點數", "最大對稱性", "最大分形維度"
]

n_features = cancer.data.shape[1]
plt.figure(figsize=(10, 8))
plt.barh(range(n_features), forest.feature_importances_, align="center")
plt.yticks(np.arange(n_features), zh_feature_names, fontproperties=my_font)
plt.xlabel("特徵重要性 (Feature Importance)", fontproperties=my_font, fontsize=12)
plt.ylabel("特徵名稱", fontproperties=my_font, fontsize=12)
plt.title("圖 2-34：乳癌資料集 - 隨機森林 (100 棵樹) 特徵重要性分佈", fontproperties=my_font, fontsize=14)
plt.ylim(-1, n_features)
plt.tight_layout()
plt.show()

#### 📌 圖 2-34 特徵重要性觀察：
與圖 2-28（單一決策樹）相比，隨機森林考慮了更多特徵（除了「最大半徑」之外，「最大凹陷點數」、「最大周長」、「最大面積」等多個特徵也都獲得了權重），這使得隨機森林對單一特徵的雜訊不會過度敏感。

### 2.3.6 梯度提升樹（Gradient Boosted Decision Trees, GBDT）

梯度提升樹是另一種經典的決策樹集成模型。與隨機森林平行建造不同，GBDT 是**循序（Sequential）建立決策樹**，每一棵樹都努力去**修正前一棵樹留下的預測殘差與錯誤**。

GBDT 通常使用**深度很淺的決策樹（如 `max_depth = 1 ~ 5`）**，配合**學習率（`learning_rate`）**控制更新步長，能打造出泛化能力極強的模型：

In [ ]:
# GBDT 分類器實作與調參 (max_depth=1)
from sklearn.ensemble import GradientBoostingClassifier

# 預設 GBDT (max_depth=3, learning_rate=0.1)
gbrt = GradientBoostingClassifier(random_state=0)
gbrt.fit(X_train, y_train)
print(f"預設 GBDT 測試集準確率：{gbrt.score(X_test, y_test):.3f}")

# 預剪枝: 限制 max_depth=1
gbrt_depth1 = GradientBoostingClassifier(random_state=0, max_depth=1)
gbrt_depth1.fit(X_train, y_train)
print(f"預剪枝 (max_depth=1) 測試集準確率：{gbrt_depth1.score(X_test, y_test):.3f}")

### 2.3.7 梯度提升樹的特徵重要性（📌 對應圖 2-35）

繪製 `max_depth=1` 的 GBDT 特徵重要性直方圖（📌 **對應圖 2-35**）：

In [ ]:
# 📌 標註：圖 2-35 —— 乳癌資料集 GBDT 特徵重要性分佈圖
n_features = cancer.data.shape[1]
plt.figure(figsize=(10, 8))
plt.barh(range(n_features), gbrt_depth1.feature_importances_, align="center")
plt.yticks(np.arange(n_features), zh_feature_names, fontproperties=my_font)
plt.xlabel("特徵重要性 (Feature Importance)", fontproperties=my_font, fontsize=12)
plt.ylabel("特徵名稱", fontproperties=my_font, fontsize=12)
plt.title("圖 2-35：乳癌資料集 - 梯度提升樹 (GBDT, max_depth=1) 特徵重要性分佈", fontproperties=my_font, fontsize=14)
plt.ylim(-1, n_features)
plt.tight_layout()
plt.show()

### 2.3.8 隨機森林 vs 梯度提升樹 優缺點與實務調參指南總結

#### 1. 隨機森林（Random Forest）
* **優勢**：強大且穩健，幾乎**不需要調參**就能獲得優異結果；可透過多執行緒（`n_jobs=-1`）高度平行運算。
* **調參建議**：主要調整 `n_estimators`（樹的數量，越多越好），`max_features` 通常使用預設值（分類任務用 $\sqrt{\text{n\_features}}$）。

#### 2. 梯度提升樹（GBDT）
* **優勢**：通常能獲得比隨機森林**更高的預測精準度**，且模型更小、預測速度更快。
* **調參建議**：主要調整 `n_estimators` 與 `learning_rate`（兩者互相平衡，低學習率搭配更多樹）；通常限制 `max_depth` 在 1~5 之間。